
### References

*   [https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876](https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876)
*   [https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo](https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo)
*   [https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/](https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/)
*   https://www.kaggle.com/code/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch

# 1. Qwen2.5 32B GPTQ Int4 Inference

In [1]:
! mkdir -p /tmp/src

In [2]:
%%writefile /tmp/src/infer_qwen.py

import os
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import torch
import vllm
import numpy as np
from vllm.lora.request import LoRARequest
import argparse
from scipy.special import softmax
df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")

MODEL_NAME = "/kaggle/input/qwen2-5-32b-instruct-gptq-int4"
LORA_PATH = "/kaggle/input/jigsaw-exp003-fold0/trained_model"
if __name__=='__main__':
    os.environ["VLLM_USE_V1"] = "0"

    llm = vllm.LLM(
        MODEL_NAME,
        # quantization='awq',
        quantization='gptq',
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.95,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=4096,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
    )
    tokenizer = llm.get_tokenizer()
    SYS_PROMPT = """
    You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
    """
    
    prompts = []
    for i, row in df.iterrows():
        text = f"""
    r/{row.subreddit}
    Rule: {row.rule}

    <POSITIVE_EXAMPLE>
    1) {row.positive_example_1}
    Violation: Yes
    
    2) {row.positive_example_2}
    Violation: Yes
    </POSITIVE_EXAMPLE>
    
    <NEGATIVE_EXAMPLE>
    1) {row.negative_example_1}
    Violation: No
    
    2) {row.negative_example_2}
    Violation: No
    </NEGATIVE_EXAMPLE>
    
    <COMMENT>
    {row.body}
    </COMMENT>

    """
        
        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]
    
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)
    
    df["prompt"] = prompts
    
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )
    logprobs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
    df = pd.concat([df, logit_matrix], axis=1)
    
    df[['Yes',"No"]] = df[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
    df["pred"] = df["Yes"]
    df['rule_violation'] = df["pred"]
    df[['row_id', 'rule_violation']].to_csv("submission_qwen.csv",index=False)
    pd.read_csv('submission_qwen.csv')

Writing /tmp/src/infer_qwen.py


In [3]:
%cd /tmp
!python src/infer_qwen.py

/tmp
2025-08-14 16:29:54.153801: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755188994.508677      57 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755188994.616576      57 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 08-14 16:30:09 [__init__.py:235] Automatically detected platform cuda.
INFO 08-14 16:30:26 [config.py:1604] Using max model len 4096
WARNING 08-14 16:30:27 [config.py:1084] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 08-14 16:30:28 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor c

# 2. Llama3.1 8B Instruct Inference

In [4]:
import os, math, numpy as np
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

In [5]:
import pandas as pd
import numpy as np

test = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
sub = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv', index_col='row_id')
sub


,rule_violation
row_id,
2029,0.5
2030,0.5
2031,0.5
2032,0.5
2033,0.5
2034,0.5
2035,0.5
2036,0.5
2037,0.5


In [6]:
import vllm

llm = vllm.LLM(
    "/kaggle/input/jigsaw-llama3-1-8b-instruct-training-one-epoch/llama-8b-instruct-jigsaw",
    tensor_parallel_size=2, 
    gpu_memory_utilization=0.95, 
    trust_remote_code=True,
    dtype="half", 
    enforce_eager=True,
    max_model_len=2048,
    # disable_log_stats=True,
    # enable_prefix_caching=True,
    
)
tokenizer = llm.get_tokenizer()


2025-08-14 16:36:20.278737: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755189380.302190      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755189380.309180      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 08-14 16:36:27 [__init__.py:235] Automatically detected platform cuda.
INFO 08-14 16:36:43 [config.py:1604] Using max model len 2048
WARNING 08-14 16:36:43 [arg_utils.py:1690] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 08-14 16:36:44 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 08-14 16:36:44 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/input/jigsaw-llama3-1-8b-instruct-training-one-epoch/llama-8b-instruct-jigsaw', speculative_config=None, tokenizer='/kaggle/input/jigsaw-llama3-1-8b-instruct-training-one-epoch/llama-8b-instruct-jigsaw', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeli

2025-08-14 16:36:49.483469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755189409.504669     402 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755189409.511198     402 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 08-14 16:36:55 [__init__.py:235] Automatically detected platform cuda.
(VllmWorkerProcess pid=402) INFO 08-14 16:36:55 [multiproc_worker_utils.py:226] Worker ready; awaiting tasks
(VllmWorkerProcess pid=402) INFO 08-14 16:36:56 [cuda.py:346] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
(VllmWorkerProcess pid=402) INFO 08-14 16:36:56 [cuda.py:395] Using XFormers backend.


[W814 16:37:07.354177195 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W814 16:37:07.732618626 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W814 16:37:17.365296173 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3


INFO 08-14 16:37:27 [__init__.py:1375] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=402) INFO 08-14 16:37:27 [__init__.py:1375] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=402) INFO 08-14 16:37:27 [pynccl.py:70] vLLM is using nccl==2.26.2
INFO 08-14 16:37:27 [pynccl.py:70] vLLM is using nccl==2.26.2


[W814 16:37:27.375273306 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3


INFO 08-14 16:37:27 [custom_all_reduce_utils.py:246] reading GPU P2P access cache from /root/.cache/vllm/gpu_p2p_access_cache_for_0,1.json
(VllmWorkerProcess pid=402) INFO 08-14 16:37:27 [custom_all_reduce_utils.py:246] reading GPU P2P access cache from /root/.cache/vllm/gpu_p2p_access_cache_for_0,1.json
INFO 08-14 16:37:27 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[1], buffer_handle=(1, 4194304, 6, 'psm_08dcccc4'), local_subscribe_addr='ipc:///tmp/cadf5ec8-b6cb-4b97-87dd-4b487325d754', remote_subscribe_addr=None, remote_addr_ipv6=False)
INFO 08-14 16:37:27 [parallel_state.py:1102] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(VllmWorkerProcess pid=402) INFO 08-14 16:37:27 [parallel_state.py:1102] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, TP rank 1, EP rank 1
INFO 08-14 16:37:27 [model_runner.py:1083] Starting to load model /kaggle/input/jigsaw-llama3-1-8b-instruct-training-one-epoch/l

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 08-14 16:39:15 [default_loader.py:262] Loading weights took 106.71 seconds
(VllmWorkerProcess pid=402) INFO 08-14 16:39:15 [default_loader.py:262] Loading weights took 106.95 seconds
INFO 08-14 16:39:16 [model_runner.py:1115] Model loading took 7.5123 GiB and 106.957389 seconds
(VllmWorkerProcess pid=402) INFO 08-14 16:39:16 [model_runner.py:1115] Model loading took 7.5123 GiB and 107.194127 seconds
(VllmWorkerProcess pid=402) INFO 08-14 16:39:20 [worker.py:295] Memory profiling takes 3.75 seconds
(VllmWorkerProcess pid=402) INFO 08-14 16:39:20 [worker.py:295] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.95) = 14.00GiB
(VllmWorkerProcess pid=402) INFO 08-14 16:39:20 [worker.py:295] model weights take 7.51GiB; non_torch_memory takes 0.12GiB; PyTorch activation peak memory takes 0.15GiB; the rest of the memory reserved for KV Cache is 6.22GiB.
INFO 08-14 16:39:20 [worker.py:295] Memory profiling takes 3.86 seconds
INFO 08-14 16:39:20 [wor

In [7]:
from typing import Any, Dict, List
from transformers import LogitsProcessor
import torch

choices = ["No", "Yes"]

KEEP = []
for x in choices:
    c = tokenizer.encode(x,add_special_tokens=False)[0]
    KEEP.append(c)
print(f"Force predictions to be tokens {KEEP} which are {choices}.")

class DigitLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer):
        self.allowed_ids = KEEP
        
    def __call__(self, input_ids: List[int], scores: torch.Tensor) -> torch.Tensor:
        scores[self.allowed_ids] += 100
        return scores

Force predictions to be tokens [2822, 9642] which are ['No', 'Yes'].


In [8]:


sys_prompt = '''You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''



In [9]:


def formatting(dataset):
    texts = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
    return texts



In [10]:


template = """
Subreddit: r/{subreddit}
Rule: {rule}
<POSITIVE_EXAMPLE>
1) {positive_example_1}
Violation: Yes

2) {positive_example_2}
Violation: Yes
</POSITIVE_EXAMPLE>
<NEGATIVE_EXAMPLE>
1) {negative_example_2}
Violation: No

2) {negative_example_1}
Violation: No
</NEGATIVE_EXAMPLE>
<COMMENT>
{body}
</COMMENT>
Violation: """



In [11]:
dataset = []
for index,row in test.iterrows():
    
    formatted_sample = [
        {
        "role": "system",
        "content": sys_prompt
    },
       {
           "role": "user",
           "content": template.format(
               rule = row.rule,
               subreddit = row.subreddit,
               body = row.body,
               positive_example_1 = row.positive_example_1,
               negative_example_1 = row.negative_example_1,
               positive_example_2 = row.positive_example_2,
               negative_example_2 = row.negative_example_2
           )
       }]
    
    dataset.append( formatted_sample )


In [12]:
all_prompts = formatting(dataset)

In [13]:
logits_processors = [DigitLogitsProcessor(tokenizer)]
responses = llm.generate(
    all_prompts,
    vllm.SamplingParams(
        n=1,  # Number of output sequences to return for each prompt.
        top_p=0.9,  # Float that controls the cumulative probability of the top tokens to consider.
        temperature=0,  # randomness of the sampling
        seed=777, # Seed for reprodicibility
        skip_special_tokens=True,  # Whether to skip special tokens in the output.
        max_tokens=1,  # Maximum number of tokens to generate per output sequence.
        logits_processors=logits_processors,
        logprobs = 2
    ),
    use_tqdm = True
)

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [14]:
results = []
errors = 0

for i,response in enumerate(responses):
    try:
        x = response.outputs[0].logprobs[0]
        logprobs = []
        for k in KEEP:
            if k in x:
                logprobs.append( math.exp(x[k].logprob) )
            else:
                logprobs.append( 0 )
                print(f"bad logits {i}")
        logprobs = np.array( logprobs )
        logprobs /= logprobs.sum()
        results.append( logprobs )
    except:
        #print(f"error {i}")
        results.append( np.array([1/2., 1/2.]) )
        errors += 1
        
print(f"There were {errors} inference errors out of {i+1} inferences")
results = np.vstack(results)

There were 0 inference errors out of 10 inferences


In [15]:
probs = [x[1] for x in results]
sub['rule_violation'] = probs
sub.to_csv('submission_llama.csv')

# 3. ENSEMBLE RESULT

In [16]:
import pandas as pd
q = pd.read_csv('submission_qwen.csv')
l = pd.read_csv('submission_llama.csv')

rq = q['rule_violation'].rank(method='average') / (len(q)+1)
rl = l['rule_violation'].rank(method='average') / (len(l)+1)

blend = 0.55*rq + 0.45*rl   # or tune the rank-weights with a tiny grid using OOF
q['rule_violation'] = blend
q.to_csv('/kaggle/working/submission.csv', index=False)